In [ ]:
import os
import shutil

# 定义路径
src_path = '/kaggle/input/laincode-fix'
dest_path = '/kaggle/working/LAIN'

# 复制文件夹（如果目标已存在先删除，方便重复调试）
if os.path.exists(dest_path):
    shutil.rmtree(dest_path)
shutil.copytree(src_path, dest_path)

# 切换工作目录
os.chdir(dest_path)
print(f"当前工作目录: {os.getcwd()}")
!ls -l

In [ ]:
import os

# 定义路径
input_images = '/kaggle/input/hico-20160224-det/hico_20160224_det/hico_20160224_det/images'
uploaded_jsons = '/kaggle/input/datajson' 
working_dir = '/kaggle/working/hicodet'

# 1. 【核心修正】先创建根目录，再拷贝文件
# -p 参数确保如果目录已存在不会报错，不存在则自动创建
!mkdir -p {working_dir}

# 2. 拷贝 JSON 文件
# 使用 -f 强制覆盖，确保后台运行时不会停下来询问
!cp -f {uploaded_jsons}/*.json {working_dir}/
print(f"✅ JSON 文件已拷贝至 {working_dir}")

# 3. 创建图片目录的层级
target_img_dir = os.path.join(working_dir, 'hico_20160224_det')
os.makedirs(target_img_dir, exist_ok=True)

# 4. 建立软连接 (Symlink)
# 加上 -sf 防止因为链接已存在而报错
link_path = os.path.join(target_img_dir, 'images')
if not os.path.exists(link_path):
    !ln -sf {input_images} {link_path}
    print(f"🔗 已建立图片软连接: {link_path} -> {input_images}")
else:
    print("🔗 软连接已存在，跳过。")

print("\n✨ 最终目录结构检查：")
!ls -F {working_dir}
!ls -F {target_img_dir}

In [ ]:
# 1. 检查并安装核心依赖
# 建议先尝试不强制覆盖 torch，如果报错再运行下面这行
# !pip install torch==2.4.1 torchvision==0.19.1 torchaudio==2.4.1 

!pip install matplotlib tqdm scipy regex ftfy wandb gdown

# 2. 安装项目特定的扩展库 pocket
# 确保你在 /kaggle/working/LAIN 目录下
import os
os.chdir('/kaggle/working/LAIN/LAIN-main') 

# 进入 pocket 目录安装
%cd pocket
!pip install -e .

# 返回 LAIN 主目录
%cd ..

In [ ]:
# 假设找出的文件路径是 /kaggle/working/LAIN/LAIN-main/CLIP/bpe_simple_vocab_16e6.txt
import os

target_path = '/kaggle/working/LAIN/LAIN-main/CLIP/bpe_simple_vocab_16e6.txt.gz'
source_path = '/kaggle/working/LAIN/LAIN-main/CLIP/bpe_simple_vocab_16e6.txt'

if os.path.exists(source_path) and not os.path.exists(target_path):
    os.rename(source_path, target_path)
    print("已修复：文件已重命名为 .txt.gz")

In [ ]:
import gzip
import shutil
import os

bpe_path = '/kaggle/working/LAIN/LAIN-main/CLIP/bpe_simple_vocab_16e6.txt.gz'

# 1. 检查文件内容
with open(bpe_path, 'rb') as f:
    magic_number = f.read(2)
    print(f"文件开头字节: {magic_number}")

# 2. 如果不是 gzip 格式 (gzip 的 magic number 是 b'\x1f\x8b')
if magic_number != b'\x1f\x8b':
    print("检测到非 Gzip 格式，正在尝试修复...")
    
    # 先备份原始文件为临时文本
    tmp_txt = bpe_path + ".tmp"
    shutil.move(bpe_path, tmp_txt)
    
    # 将文本内容重新压缩为真正的 .gz
    with open(tmp_txt, 'rb') as f_in:
        with gzip.open(bpe_path, 'wb') as f_out:
            shutil.copyfileobj(f_in, f_out)
    
    os.remove(tmp_txt)
    print("✅ 修复完成！文件已重新压缩为标准 Gzip 格式。")
else:
    print("文件格式似乎正确，无需修复。")

In [ ]:
import os
import torch
import gc

# 1. 定位工作目录
BASE_PATH = "/kaggle/working/LAIN/LAIN-main"
if os.path.exists(BASE_PATH):
    os.chdir(BASE_PATH)
    print("🔧 [Step 1] 正在执行底层代码维度修正...")

    # --- A. 模型结构深度修复 (models/LAIN.py 及其他) ---
    # 强制将模型定义中的 LayerNorm(768) 改为 512
    !find ./models -name "*.py" | xargs sed -i 's/768/512/g'
    # 关键：将 Transformer 头数从 12 改为 8 (解决 Index Out of Bounds)
    !find ./models -name "*.py" | xargs sed -i 's/nheads=12/nheads=8/g'
    !find ./models -name "*.py" | xargs sed -i 's/num_heads=12/num_heads=8/g'

    # --- B. 参数配置修复 (utils/args.py) ---
    args_file = "utils/args.py"
    if os.path.exists(args_file):
        # 针对不同写法的 default 值进行全面替换
        !sed -i "s/default=768, type=int/default=512, type=int/g" {args_file}
        !sed -i "s/default=12, type=int/default=8, type=int/g" {args_file} 
        !sed -i "s/clip_text_transformer_heads_vit', default=12/clip_text_transformer_heads_vit', default=8/g" {args_file}
        !sed -i "s/clip_text_transformer_width_vit', default=768/clip_text_transformer_width_vit', default=512/g" {args_file}
        # 修改 Patch Size (ViT-B通常是16)
        !sed -i "s/patch_size_vit', default=14/patch_size_vit', default=16/g" {args_file}

    # --- C. PyTorch 2.6+ 兼容性修复 ---
    !find . -name "*.py" | xargs sed -i 's/, weights_only=False//g'
    !find . -name "*.py" | xargs sed -i 's/torch.load(\([^)]*\))/torch.load(\1, weights_only=False)/g'

    # --- D. 显存清理 ---
    !fuser -v /dev/nvidia* 2>/dev/null | awk '{print $2}' | xargs -r kill -9
    gc.collect()
    torch.cuda.empty_cache()
    
    print("✅ 底层维度与兼容性已修复。请继续执行 Step 2。")
else:
    print(f"❌ 找不到路径: {BASE_PATH}")

In [ ]:
!python /kaggle/working/LAIN/LAIN-main/compare_adapter.py